In [1]:
import torch

class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(

            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),

            torch.nn.Linear(30,20),
            torch.nn.ReLU(),

            torch.nn.Linear(20,num_outputs)
        )

    def forward(self,x):
        logits = self.layers(x)
        return logits

model = NeuralNetwork(50,3)
print(model)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("훈련 가능한 모델의 총 파라미터 수 : ", num_params)
print(model.layers[0].weight)
print(model.layers[0].weight.shape)
print(model.layers[0].bias)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)
훈련 가능한 모델의 총 파라미터 수 :  2213
Parameter containing:
tensor([[-0.1309,  0.0483, -0.0947,  ...,  0.0954,  0.0323, -0.0119],
        [-0.0801, -0.1212,  0.1344,  ...,  0.1155,  0.1160, -0.0275],
        [ 0.0434, -0.0674,  0.0491,  ...,  0.0360,  0.0463,  0.0225],
        ...,
        [ 0.1307,  0.0886,  0.1268,  ..., -0.0603, -0.0746,  0.0396],
        [-0.1101,  0.0575, -0.0617,  ...,  0.0240,  0.0264,  0.1374],
        [ 0.0602,  0.0613,  0.0432,  ..., -0.0082, -0.0142, -0.0347]],
       requires_grad=True)
torch.Size([30, 50])
Parameter containing:
tensor([ 9.3723e-02, -9.6809e-02,  1.0522e-01,  3.5042e-02,  1.1559e-01,
        -1.4117e-01, -1.0126e-01, -9.2367e-02,  1.4513e-02,  5.8585e-02,
        -1.1720e-01,  7.0759e-02, -9.9336e-0

In [2]:
torch.manual_seed(123)
x = torch.rand((1,50))
print(x)
out = model(x)
print(out) # grad_fn=<AddmmBackward0> : 계산 그래프에서 변수를 계산하는데 마지막으로 사용된 함수, 이 경우 행렬 곱셈과 덧셈 연산으로 만들어짐, 뒤에서 앞으로 읽어야 됨. Addmm → 곱셈 후 덧셈

tensor([[0.2961, 0.5166, 0.2517, 0.6886, 0.0740, 0.8665, 0.1366, 0.1025, 0.1841,
         0.7264, 0.3153, 0.6871, 0.0756, 0.1966, 0.3164, 0.4017, 0.1186, 0.8274,
         0.3821, 0.6605, 0.8536, 0.5932, 0.6367, 0.9826, 0.2745, 0.6584, 0.2775,
         0.8573, 0.8993, 0.0390, 0.9268, 0.7388, 0.7179, 0.7058, 0.9156, 0.4340,
         0.0772, 0.3565, 0.1479, 0.5331, 0.4066, 0.2318, 0.4545, 0.9737, 0.4606,
         0.5159, 0.4220, 0.5786, 0.9455, 0.8057]])
tensor([[-0.0363, -0.1368,  0.0373]], grad_fn=<AddmmBackward0>)


In [3]:
with torch.no_grad() : #훈련이나 역전파 없이 신경망을 사용만 할 때 (훈련 이후 예측할 때 등), 계산 그래프 구성을 하지 않음
    out = model(x)
print(out)

tensor([[-0.0363, -0.1368,  0.0373]])


In [4]:
x_train = torch.tensor([
    [-1.2, 3.1], 
    [-0.9, 2.9], 
    [-0.5, 2.6], 
    [2.3, -1.1], 
    [2.7, -1.5]
    ])
y_train = torch.tensor([0, 0, 0, 1, 1])

x_test = torch.tensor([[-0.8, 2.8],[2.6,-1.6]])
y_test = torch.tensor([0,1])


from torch.utils.data import Dataset, DataLoader

class ToyDataset(Dataset):
    def __init__(self, x,y):
        self.features = x
        self.labels = y

    def __getitem__(self, index): # 정확히 하나의 데이터 레코드와 이에 해당하는 레이블 추출
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y

    def __len__(self): # 데이터셋의 총길이 반환
        return self.labels.shape[0]

train_ds = ToyDataset(x_train, y_train)
test_ds = ToyDataset(x_test, y_test)

In [5]:
print(len(train_ds))

5


In [6]:
torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers = 0
)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=0
)


for idx, (x,y) in enumerate(train_loader):
    print(f"배치 {idx+1}: ", x,y)

배치 1:  tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
배치 2:  tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
배치 3:  tensor([[ 2.7000, -1.5000]]) tensor([1])


In [7]:
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    drop_last = True, # 마지막 배치를 버림 (마지막 배치에 샘플이 배치 크기 만큼 없을 수도 있기 때문에)
)

for idx, (x,y) in enumerate(train_loader):
    print(f"배치 {idx+1}: ", x,y)

배치 1:  tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
배치 2:  tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])


In [8]:
import torch.nn.functional as F

torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2) # 2개의 feature, 2개의 label
optimizer = torch.optim.SGD(model.parameters(), lr=0.5) # 옵티마이저에 최적화할 파라미터 전달

num_epochs = 3
for epoch in range(num_epochs):
    model.train() # dropout이나 Batch normailzation 적용 

    for batch_idx, (features, labels) in enumerate(train_loader):
        logits = model(features)
        loss = F.cross_entropy(logits, labels)
        optimizer.zero_grad() #이전 반복에서 구한 그레이디언트를 0으로 지정하여 의도치 않게 그레이디언트가 누적되지 않게 함
        loss.backward() # 모델 파라미터에 대한 손실 그레이디언트를 계산
        optimizer.step() # 옵티마이저가 그레이디언트를 사용해 모델 파라미터를 업뎃

        print(f"에포크 : {epoch+1:03d}/{num_epochs:03d}"
              f" | 배치 {batch_idx:03d}/{len(train_loader):03d}"
              f" | 훈련 손실: {loss: .2f}")

    model.eval() # dropout이나 Batch normailzation 적용 X

에포크 : 001/003 | 배치 000/002 | 훈련 손실:  0.75
에포크 : 001/003 | 배치 001/002 | 훈련 손실:  0.65
에포크 : 002/003 | 배치 000/002 | 훈련 손실:  0.44
에포크 : 002/003 | 배치 001/002 | 훈련 손실:  0.13
에포크 : 003/003 | 배치 000/002 | 훈련 손실:  0.03
에포크 : 003/003 | 배치 001/002 | 훈련 손실:  0.00


In [9]:
model.eval()
with torch.no_grad():
    outputs = model(x_train)
print(outputs)
torch.set_printoptions(sci_mode=False)
probas = torch.softmax(outputs, dim=1)
print(probas)

tensor([[ 2.8569, -4.1618],
        [ 2.5382, -3.7548],
        [ 2.0944, -3.1820],
        [-1.4814,  1.4816],
        [-1.7176,  1.7342]])
tensor([[0.9991, 0.0009],
        [0.9982, 0.0018],
        [0.9949, 0.0051],
        [0.0491, 0.9509],
        [0.0307, 0.9693]])


In [10]:
predictions = torch.argmax(outputs, dim=1)
print(predictions)
predictions == y_train

tensor([0, 0, 0, 1, 1])


tensor([True, True, True, True, True])

In [ ]:
def compute_accuracy(model, dataloader, device='cuda'):
    model = model.eval()
    correct = 0.0
    total_examples = 0

    for idx, (features, labels) in enumerate(dataloader):
        features, labels = features.to(device), labels.to(device)

        with torch.no_grad():
            logits = model(features)
        predictions = torch.argmax(logits, dim=1)
        compare = labels == predictions
        correct += torch.sum(compare)
        total_examples += len(compare)
    return (correct / total_examples).item()

1.0
1.0


In [12]:
torch.save(model.state_dict(), "simple_model.pth")
model = NeuralNetwork(2,2)
model.load_state_dict(torch.load("simple_model.pth"))

<All keys matched successfully>

In [16]:
# CPU
tensor_1 = torch.tensor([1., 2., 3.])
tensor_2 = torch.tensor([4., 5., 6.])
print(tensor_1 + tensor_2)

#GPU
tensor_1 = tensor_1.to("cuda")
tensor_2 = tensor_2.to("cuda")
print(tensor_1+tensor_2)

tensor([5., 7., 9.])
tensor([5., 7., 9.], device='cuda:0')


In [19]:
torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr = 0.5)
num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):
        features, labels = features.to(device), labels.to(device)
        logits = model(features)
        loss = F.cross_entropy(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


        print(f"에포크 : {epoch+1:03d}/{num_epochs:03d}"
              f" | 배치 {batch_idx:03d}/{len(train_loader):03d}"
              f" | 훈련 손실: {loss: .2f}")

    model.eval()

에포크 : 001/003 | 배치 000/002 | 훈련 손실:  0.75
에포크 : 001/003 | 배치 001/002 | 훈련 손실:  0.65
에포크 : 002/003 | 배치 000/002 | 훈련 손실:  0.44
에포크 : 002/003 | 배치 001/002 | 훈련 손실:  0.13
에포크 : 003/003 | 배치 000/002 | 훈련 손실:  0.03
에포크 : 003/003 | 배치 001/002 | 훈련 손실:  0.00


In [ ]:
import torch.multiprocessing as mp
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.distributed import init_process_group, destroy_process_group
import os
def ddp_setup(rank, world_size):
    os.environ["MASTER_ADDR"] = 'localhost' #메인 노드 주소
    os.environ["MASTER_PORT"] = "12345" # 사용 가능한 임의의 포트
    init_process_group(
        backend="nccl", 
        rank=rank, # 사용할 GPU 인덱스
        world_size = world_size # 사용할 GPU 개수
    )
    torch.cuda.set_device(rank) # 텐서가 할당되고 연산을 수행할 현재 GPU 장치 설정

def prepare_dataset():
    x_train = torch.tensor([
        [-1.2, 3.1],
        [-0.9, 2.9],
        [-0.5, 2.6],
        [2.3, -1.1],
        [2.7, -1.5]
    ])
    y_train = torch.tensor([0, 0, 0, 1, 1])

    x_test = torch.tensor([
        [-0.8, 2.8],
        [2.6, -1.6],
    ])
    y_test = torch.tensor([0, 1])
    train_loader = DataLoader(
        dataset=train_ds,
        batch_size=2,
        shuffle=False, # DistributedSampler가 셔플 담당
        pin_memory = True, # GPU에서 훈련할 때 메모리 전송 속도 향상
        drop_last=True,
        sampler = DistributedSampler(train_ds) # 데이터셋을 각 프로세스(GPU)
        )
    test_loader = DataLoader(
        dataset=test_ds,
        batch_size=2,
        shuffle=False
    )
    return train_loader, test_loader
def main(rank, world_size, num_epcohs):
    ddp_setup(rank, world_size)
    train_loader, test_loader = prepare_dataset()
    model = NeuralNetwork(num_inputs=2, num_outputs=2)
    model.to(rank)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
    for epoch in range(num_epochs):
        train_loader.sampler.set_epoch(epoch)
        model.train()
        for features, labels in train_loader:
            features, labels = features.to(rank), labels.to(rank)
            print(f"[GPU{rank}] Epoch : {epoch+1:03d}/{num_epcohs:03d}"
                  f"| Batch {labels.shape[0]:03d}"
                  f"| Loss: {loss:.2f}")
    model.eval()
    train_acc = compute_accuracy(model, train_loader, device=rank)
    print(f"[GPU{rank}] Train Accuracy: ", train_acc)
    test_acc = compute_accuracy(model, test_loader, device=rank)
    print(f"[GPU{rank}] Test Accuracy: ", train_acc)
    destroy_process_group()

if __name__ == "__main__":
    print("사용 가능한 GPU 개수: ", torch.cuda.device_count())
    torch.manual_seed(123)
    num_epochs=3
    world_size = torch.cuda.device_count()
    mp.spawn(main, args=(world_size, num_epochs), nprocs=world_size) # 여러 프로세스를 사용해 main 함수를 시작. nprocs=world_size는 GPU당 하나의 프로세스를 만든다는 의미
